# LightGCN и HeteroGNN

LightGCN обучается на связях пользователей и фильмов. HeteroGNN дополнительно учитывает узлы жанров, актёров, режиссёров и стран. Обе модели предсказывают релевантность через скалярное произведение эмбеддингов и обучаются с BPR-loss.

Размерность 64, три слоя, размер батча 65536. Семь недель CV: с 21 июня по 2 августа 2021 года. Тест: 9 и 16 августа. Число эпох для теста выбирается по CV. Метрики: MAP@10 и NDCG@10.

In [1]:
import pandas as pd
import numpy as np

interactions = pd.read_csv('data_original/interactions.csv', parse_dates=['last_watch_dt'])
items = pd.read_csv('data_original/items.csv').reset_index(drop=True)
item_ids = pd.Index(items['item_id'])

interactions[['user_id','item_id','last_watch_dt']].head()

,user_id,item_id,last_watch_dt
0,176549,9506,2021-05-11
1,699317,1659,2021-05-29
2,656683,7107,2021-05-09
3,864613,7638,2021-07-05
4,964868,9506,2021-04-30


In [2]:
items[['item_id','genres','actors','directors','countries']].head()

,item_id,genres,actors,directors,countries
0,10711,"драмы, зарубежные, детективы, мелодрамы","Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Педро Альмодовар,Испания
1,2508,"зарубежные, приключения, комедии","Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Скот Армстронг,США
2,10716,"криминал, зарубежные, триллеры, боевики, комедии","Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Адам П. Калтраро,Канада
3,7868,"драмы, зарубежные, мелодрамы","Александра Риддлстон-Барретт, Джеральдин Джейм...",Эндрю Хэй,Великобритания
4,16268,"драмы, спорт, советские, мелодрамы","Александр Абдулов, Александр Демьяненко, Алекс...",Виктор Садовский,СССР


In [3]:
def history_graph(t):
    h = interactions.loc[interactions["last_watch_dt"] < t, ["user_id", "item_id"]].drop_duplicates()
    users, user_ids = pd.factorize(h["user_id"], sort=True)
    products = item_ids.get_indexer(h["item_id"])
    return user_ids, users.astype(np.int32), products.astype(np.int32)

In [4]:
metadata = {}
meta_size = 0

for field in ('genres', 'actors', 'directors', 'countries'):
    values = items[field].fillna("").astype(str)
    tokens = values.map(lambda s: sorted({v.strip() for v in s.split(",") if v.strip()}))
    pairs = tokens.explode().dropna()
    labels, names = pd.factorize(pairs, sort=True)
    metadata[field] = {
        "item": pairs.index.to_numpy(dtype=np.int32),
        "attr": labels.astype(np.int32) + meta_size,
        "start": meta_size,
        "end": meta_size + len(names),
    }
    meta_size += len(names)
    print(f"{field}: {len(names):,} узлов, {len(pairs):,} рёбер")

print('Всего узлов категорий:', f'{meta_size:,}')

genres: 95 узлов, 40,637 рёбер
actors: 61,883 узлов, 139,670 рёбер
directors: 8,838 узлов, 17,810 рёбер
countries: 91 узлов, 18,258 рёбер
Всего узлов категорий: 70,907


In [5]:
import torch
from torch import nn
from torch.nn import functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [6]:
def normalized_graph(users, products, n_users, n_items):
    u = torch.as_tensor(users.astype(np.int64), device=device)
    i = torch.as_tensor(products.astype(np.int64), device=device) + n_users
    degrees = torch.bincount(torch.cat([u, i]), minlength=n_users + n_items).float()
    weights = (degrees[u] * degrees[i]).clamp_min(1).rsqrt()
    indices = torch.stack([torch.cat([u, i]), torch.cat([i, u])])
    adj = torch.sparse_coo_tensor(
        indices, torch.cat([weights, weights]),
        (n_users + n_items, n_users + n_items), device=device
    ).coalesce()
    return adj, degrees[n_users:]

def normalized_relation(src, dst, n_nodes):
    src = torch.as_tensor(np.asarray(src, dtype=np.int64), device=device)
    dst = torch.as_tensor(np.asarray(dst, dtype=np.int64), device=device)
    deg = torch.bincount(torch.cat([src, dst]), minlength=n_nodes).float()
    weight = (deg[src] * deg[dst]).clamp_min(1).rsqrt()
    indices = torch.stack([torch.cat([src, dst]), torch.cat([dst, src])])
    return torch.sparse_coo_tensor(
        indices, torch.cat([weight, weight]), (n_nodes, n_nodes), device=device
    ).coalesce()

In [7]:
def build_graph(users, products, n_users):
    n_items = len(item_ids)
    n_nodes = n_users + n_items + meta_size
    adjs = {"interaction": normalized_relation(users, n_users + products, n_nodes)}
    presence = []
    for field, info in metadata.items():
        adjs[field] = normalized_relation(
            n_users + info["item"], n_users + n_items + info["attr"], n_nodes
        )
        counts = np.bincount(info["item"], minlength=n_items)
        presence.append((counts > 0).astype(np.float32))
    presence = torch.from_numpy(np.stack(presence)).to(device)
    degree = torch.from_numpy(np.bincount(products, minlength=n_items)).to(device)
    return adjs, presence, degree

In [8]:
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, dim=64, layers=3):
        super().__init__()
        self.emb = nn.Embedding(n_users + n_items, dim)
        nn.init.normal_(self.emb.weight, std=0.1)
        self.layers = layers

    def forward(self, adj):
        x = self.emb.weight
        result = x
        for _ in range(self.layers):
            x = torch.sparse.mm(adj, x)
            result = result + x
        return result / (self.layers + 1)

In [9]:
class HeteroLightGCN(nn.Module):
    def __init__(self, n_users, presence, dim=64, layers=3):
        super().__init__()
        self.n_users = n_users
        self.n_items = len(item_ids)
        self.layers = layers
        self.fields = list(metadata)
        self.emb = nn.Embedding(n_users + self.n_items + meta_size, dim)
        nn.init.normal_(self.emb.weight, std=0.1)
        self.relation_logits = nn.Parameter(torch.full((len(self.fields),), -1.0986123))
        self.register_buffer("presence", presence)

    def forward(self, adjs):
        x = self.emb.weight
        result = x
        a, b = self.n_users, self.n_users + self.n_items
        gates = torch.sigmoid(self.relation_logits)
        denom = 1 + (gates[:, None] * self.presence).sum(dim=0)
        for _ in range(self.layers):
            ui_msg = torch.sparse.mm(adjs["interaction"], x)
            msgs = [torch.sparse.mm(adjs[field], x) for field in self.fields]
            item_msg = ui_msg[a:b]
            for gate, msg in zip(gates, msgs):
                item_msg = item_msg + gate * msg[a:b]
            attrs = [msg[b + metadata[field]["start"]: b + metadata[field]["end"]]
                     for field, msg in zip(self.fields, msgs)]
            x = torch.cat([ui_msg[:a], item_msg / denom[:, None], *attrs])
            result = result + x
        return result / (self.layers + 1)

In [10]:
def seen_before(keys, sorted_seen):
    idx = np.searchsorted(sorted_seen, keys)
    return (idx < len(sorted_seen)) & (sorted_seen[np.minimum(idx, len(sorted_seen) - 1)] == keys)

def sample_triples(rng, users, products, present, seen, n_items, size):
    pick = rng.integers(len(users), size=size)
    u, pos = users[pick], products[pick]
    neg = rng.choice(present, size=size)
    bad = seen_before(u.astype(np.int64) * n_items + neg, seen)
    attempts = 0
    while bad.any():
        neg[bad] = rng.choice(present, size=int(bad.sum()))
        keys = u[bad].astype(np.int64) * n_items + neg[bad]
        bad[bad] = seen_before(keys, seen)
        attempts += 1
        if attempts > 100:
            raise RuntimeError("Не удалось подобрать непросмотренный объект")
    return u, pos, neg

In [11]:
def bpr_step(model, graph, opt, triples, n_users):
    u, pos, neg = [torch.as_tensor(v.astype(np.int64), device=device) for v in triples]
    opt.zero_grad(set_to_none=True)
    e = model(graph)
    scores = (e[u] * (e[n_users + pos] - e[n_users + neg])).sum(dim=1)
    loss = F.softplus(-scores).mean()
    base = model.emb.weight
    reg = (base[u].square().sum(1) + base[n_users + pos].square().sum(1)
           + base[n_users + neg].square().sum(1)).mean() * (1e-4 / 2)
    (loss + reg).backward()
    opt.step()
    return float(loss.detach())

In [12]:
def target_counts_for_week(t):
    future = interactions.loc[
        (interactions["last_watch_dt"] >= t) &
        (interactions["last_watch_dt"] < t + pd.Timedelta(days=7))
    ]
    return future.groupby("user_id")["item_id"].nunique()

def ranking_metrics(scores, labels, counts, rrf):
    counts = np.asarray(counts, dtype=np.int32)
    n = len(counts)
    scores = np.asarray(scores).reshape(n, 500)
    labels = np.asarray(labels).reshape(n, 500)
    rrf = np.asarray(rrf).reshape(n, 500)
    top = np.lexsort((rrf, -scores), axis=1)[:, :10]
    rel = np.take_along_axis(labels, top, axis=1)
    discounts = 1 / np.log2(np.arange(2, 12))
    precision = np.cumsum(rel, axis=1) / np.arange(1, 11)
    ap = (precision * rel).sum(axis=1) / np.minimum(counts, 10)
    dcg = (rel * discounts).sum(axis=1)
    ideal = np.r_[0, np.cumsum(discounts)][np.minimum(counts, 10)]
    return {"map10": float(ap.mean()), "ndcg10": float((dcg / ideal).mean())}

In [13]:
import pyarrow.parquet as pq
from pathlib import Path

replay_dir = Path('data_processed/replay')
example = pq.ParquetFile(replay_dir / '2021-06-21.parquet')
display(next(example.iter_batches(batch_size=5)).to_pandas()[['user_id','item_id','label','rrf_rank']])

,user_id,item_id,label,rrf_rank
0,2,565,0,1
1,2,10323,0,2
2,2,10761,0,3
3,2,7582,0,4
4,2,10440,0,5


In [14]:
def replay_batches(t, columns=("user_id", "item_id", "label", "rrf_rank"), batch_users=1000):
    pf = pq.ParquetFile(replay_dir / f"{t:%Y-%m-%d}.parquet")
    tail = None
    for batch in pf.iter_batches(batch_size=batch_users * 500, columns=columns):
        part = batch.to_pandas()
        if tail is not None:
            part = pd.concat([tail, part], ignore_index=True)
        stop = len(part) // 500 * 500
        if stop:
            ready = part.iloc[:stop].reset_index(drop=True)
            yield ready
        tail = part.iloc[stop:].copy() if stop < len(part) else None

In [15]:
def early_stop_sample(t, folder):
    path = folder / f"early_{t:%Y%m%d}.npz"
    if path.exists():
        return dict(np.load(path))
    pf = pq.ParquetFile(replay_dir / f"{t:%Y-%m-%d}.parquet")
    n = pf.metadata.num_rows // 500
    rng = np.random.default_rng(42 + int(t.strftime("%Y%m%d")))
    chosen = np.sort(rng.choice(n, size=min(1500, n), replace=False))
    parts = []
    offset = 0
    for part in replay_batches(t):
        groups = len(part) // 500
        local = chosen[(chosen >= offset) & (chosen < offset + groups)] - offset
        if len(local):
            positions = (local[:, None] * 500 + np.arange(500)).ravel()
            parts.append(part.iloc[positions])
        offset += groups
    df = pd.concat(parts, ignore_index=True)
    n_selected = len(df) // 500
    users = df["user_id"].to_numpy().reshape(n_selected, 500)[:, 0]
    counts = target_counts_for_week(t).reindex(users).to_numpy()
    result = dict(
        users=users, items=df["item_id"].to_numpy().reshape(-1, 500),
        labels=df["label"].to_numpy().reshape(-1, 500),
        rrf=df["rrf_rank"].to_numpy().reshape(-1, 500),
        counts=counts.astype(np.int32)
    )
    np.savez(path, **result)
    return result

In [16]:
def score_candidates(emb, user_ids, candidates, graph_user_ids, available, n_users):
    user_codes = graph_user_ids.get_indexer(user_ids)
    item_codes = item_ids.get_indexer(np.asarray(candidates).ravel()).reshape(-1, 500)
    u = torch.as_tensor(user_codes, dtype=torch.long, device=device)
    i = torch.as_tensor(item_codes, dtype=torch.long, device=device)
    with torch.no_grad():
        score = (emb[u, None, :] * emb[n_users + i]).sum(dim=-1)
        score = score.masked_fill(~available[i], -1e6)
    return score.cpu().numpy()

In [17]:
def sample_metrics(model, graph, sample, user_ids, available):
    model.eval()
    with torch.no_grad():
        e = model(graph)
        scores = score_candidates(e, sample['users'], sample['items'],
                                  user_ids, available, len(user_ids))
    return ranking_metrics(scores, sample['labels'], sample['counts'], sample['rrf'])

In [18]:
from tqdm.auto import tqdm
import gc
import json

model_dirs = {
    'LightGCN': Path('results/lightgcn_full_history'),
    'HeteroGNN': Path('results/heterolightgcn_full_history'),
}
for folder in model_dirs.values():
    folder.mkdir(parents=True, exist_ok=True)

/home/mpodsytnik/.conda/envs/avito/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
def fit_graph(t, name, user_ids, users, products, graph, presence, available, fixed_epochs=None):
    folder = model_dirs[name]
    stem = f'{t:%Y%m%d}'
    best_path = folder / f'model_{stem}.pt'
    resume_path = folder / f'resume_{stem}.pt'
    done_path = folder / f'done_{stem}.json'
    if done_path.exists() and best_path.exists():
        return json.loads(done_path.read_text())

    seed = 42 + int(t.strftime('%Y%m%d'))
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    n_users, n_items = len(user_ids), len(item_ids)
    if name == 'LightGCN':
        model = LightGCN(n_users, n_items).to(device)
    else:
        model = HeteroLightGCN(n_users, presence).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.005,
                           weight_decay=1e-6 if name == 'HeteroGNN' else 0)
    present = np.unique(products)
    seen = np.unique(users.astype(np.int64) * n_items + products)
    sample = early_stop_sample(t, folder) if fixed_epochs is None else None
    best_metric, best_epoch, stale, start = -np.inf, 0, 0, 1
    max_epochs = 35 if fixed_epochs is None else int(fixed_epochs)

    if resume_path.exists():
        state = torch.load(resume_path, map_location=device, weights_only=False)
        if state['fixed_epochs'] == fixed_epochs:
            model.load_state_dict(state['model'])
            opt.load_state_dict(state['optimizer'])
            rng.bit_generator.state = state['rng']
            start = state['epoch'] + 1
            best_metric, best_epoch = state['best_metric'], state['best_epoch']
            stale = state['stale']

    for epoch in tqdm(range(start, max_epochs + 1), desc=f'{name} {t.date()}'):
        model.train()
        losses = []
        for _ in range(3):
            triples = sample_triples(rng, users, products, present, seen, n_items, 65536)
            losses.append(bpr_step(model, graph, opt, triples, n_users))
        if fixed_epochs is None and epoch % 5 == 0:
            metric = sample_metrics(model, graph, sample, user_ids, available)['map10']
            print(f'Эпоха {epoch}: BPR={np.mean(losses):.4f}, MAP@10={metric:.5f}')
            if metric > best_metric + 1e-6:
                best_metric, best_epoch, stale = metric, epoch, 0
                torch.save({'model': model.state_dict(), 'epoch': epoch}, best_path)
            else:
                stale += 1
        if epoch % 5 == 0 or epoch == max_epochs:
            torch.save(dict(model=model.state_dict(), optimizer=opt.state_dict(),
                             rng=rng.bit_generator.state, epoch=epoch,
                             best_metric=best_metric, best_epoch=best_epoch,
                             stale=stale, fixed_epochs=fixed_epochs), resume_path)
        if fixed_epochs is None and stale >= 3:
            break

    if fixed_epochs is not None:
        best_epoch = int(fixed_epochs)
        torch.save({'model': model.state_dict(), 'epoch': best_epoch}, best_path)
    info = {'date': str(t.date()), 'best_epoch': best_epoch,
            'early_map10': None if fixed_epochs is not None else float(best_metric),
            'n_users': n_users, 'n_edges': len(users), 'fixed_epochs': fixed_epochs}
    if name == 'HeteroGNN':
        info['gates'] = dict(zip(tuple(metadata), torch.sigmoid(model.relation_logits).detach().cpu().tolist()))
    done_path.write_text(json.dumps(info, indent=2))
    resume_path.unlink(missing_ok=True)
    del model, opt
    gc.collect()
    return info

In [20]:
def score_week(t, model, graph, available, user_ids, name):
    counts_by_user = target_counts_for_week(t)
    total = {m: {'map10': 0.0, 'ndcg10': 0.0} for m in (name, 'RRF')}
    n_total = 0
    model.eval()
    with torch.no_grad():
        emb = model(graph)
        for part in tqdm(replay_batches(t), desc=f'Оценка {name} {t.date()}'):
            n = len(part) // 500
            u = part['user_id'].to_numpy()[::500]
            movies = part['item_id'].to_numpy().reshape(n, 500)
            labels = part['label'].to_numpy().reshape(n, 500)
            rrf = part['rrf_rank'].to_numpy().reshape(n, 500)
            counts = counts_by_user.reindex(u).to_numpy()
            score = score_candidates(emb, u, movies, user_ids, available, len(user_ids))
            for m, values in ((name, score), ('RRF', -rrf)):
                metrics = ranking_metrics(values, labels, counts, rrf)
                for key in ('map10', 'ndcg10'):
                    total[m][key] += metrics[key] * n
            n_total += n
    return [dict(week=str(t.date()), model=m, val_groups=n_total,
                 **{key: value / n_total for key, value in metrics.items()})
            for m, metrics in total.items()]

In [21]:
def run_week(t, name, fixed_epochs=None):
    user_ids, users, products = history_graph(t)
    n_users = len(user_ids)
    if name == 'LightGCN':
        graph, degree = normalized_graph(users, products, n_users, len(item_ids))
        presence, available = None, degree > 0
    else:
        graph, presence, degree = build_graph(users, products, n_users)
        available = (degree > 0) | presence.bool().any(dim=0)
    info = fit_graph(t, name, user_ids, users, products, graph, presence, available,
                     fixed_epochs=fixed_epochs)
    if name == 'LightGCN':
        model = LightGCN(n_users, len(item_ids)).to(device)
    else:
        model = HeteroLightGCN(n_users, presence).to(device)
    checkpoint = torch.load(model_dirs[name] / f'model_{t:%Y%m%d}.pt',
                            map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    rows = score_week(t, model, graph, available, user_ids, name)
    for row in rows:
        row['best_epoch'] = info['best_epoch']
    del model, checkpoint, graph, users, products, user_ids, degree, presence, available
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return rows

In [22]:
cv_dates = pd.date_range('2021-06-21', periods=7, freq='7D')

for name, folder in model_dirs.items():
    cv_file = folder / 'cv_results.csv'
    results = pd.read_csv(cv_file).to_dict('records') if cv_file.exists() else []
    for t in cv_dates:
        ready = {r['model'] for r in results if r['week'] == str(t.date())}
        if {name, 'RRF'} <= ready:
            continue
        rows = run_week(t, name)
        results = [r for r in results if r['week'] != str(t.date())] + rows
        pd.DataFrame(results).to_csv(cv_file, index=False)
        display(pd.DataFrame(rows))

    display(pd.DataFrame(results).query('model == @name').set_index('week')[['map10','ndcg10']].round(5))

LightGCN 2021-06-21:  11%|█▏        | 4/35 [00:04<00:22,  1.38it/s]

Эпоха 5: BPR=0.4308, MAP@10=0.11697


LightGCN 2021-06-21:  26%|██▌       | 9/35 [00:06<00:11,  2.17it/s]

Эпоха 10: BPR=0.2002, MAP@10=0.12117


LightGCN 2021-06-21:  40%|████      | 14/35 [00:09<00:08,  2.42it/s]

...

HeteroGNN 2021-08-02:  97%|█████████▋| 34/35 [02:09<00:03,  3.63s/it]

Эпоха 35: BPR=0.2290, MAP@10=0.09206


HeteroGNN 2021-08-02: 100%|██████████| 35/35 [02:14<00:00,  3.85s/it]
Оценка HeteroGNN 2021-08-02: 100it [00:04, 21.11it/s]


,week,model,val_groups,map10,ndcg10,best_epoch
0,2021-08-02,HeteroGNN,99400,0.087664,0.123577,30
1,2021-08-02,RRF,99400,0.054630,0.087914,30


,map10,ndcg10
week,,
2021-06-21,0.11801,0.16175
2021-06-28,0.15747,0.24570
2021-07-05,0.14891,0.20270
2021-07-12,0.11129,0.15624
2021-07-19,0.09494,0.13727
2021-07-26,0.08961,0.12801
2021-08-02,0.08766,0.12358


In [23]:
for name, folder in model_dirs.items():
    cv = pd.read_csv(folder / 'cv_results.csv')
    epochs = cv.loc[cv.model == name, 'best_epoch']
    test_epochs = max(5, int(np.median(epochs)))
    test_file = folder / 'test_results.csv'
    results = pd.read_csv(test_file).to_dict('records') if test_file.exists() else []

    for t in pd.to_datetime(['2021-08-09', '2021-08-16']):
        ready = {r['model'] for r in results if r['week'] == str(t.date())}
        if {name, 'RRF'} <= ready:
            continue
        rows = run_week(t, name, fixed_epochs=test_epochs)
        results = [r for r in results if r['week'] != str(t.date())] + rows
        pd.DataFrame(results).to_csv(test_file, index=False)
        display(pd.DataFrame(rows))

LightGCN 2021-08-09: 100%|██████████| 10/10 [00:07<00:00,  1.28it/s]
Оценка LightGCN 2021-08-09: 106it [00:05, 19.36it/s]


,week,model,val_groups,map10,ndcg10,best_epoch
0,2021-08-09,LightGCN,105212,0.081176,0.115395,10
1,2021-08-09,RRF,105212,0.049992,0.081822,10


LightGCN 2021-08-16: 100%|██████████| 10/10 [00:08<00:00,  1.19it/s]
Оценка LightGCN 2021-08-16: 112it [00:05, 19.59it/s]


,week,model,val_groups,map10,ndcg10,best_epoch
0,2021-08-16,LightGCN,111240,0.077606,0.111173,10
1,2021-08-16,RRF,111240,0.047603,0.079496,10


HeteroGNN 2021-08-09: 100%|██████████| 30/30 [01:57<00:00,  3.93s/it]
Оценка HeteroGNN 2021-08-09: 106it [00:05, 19.56it/s]


,week,model,val_groups,map10,ndcg10,best_epoch
0,2021-08-09,HeteroGNN,105212,0.081650,0.115941,30
1,2021-08-09,RRF,105212,0.049992,0.081822,30


HeteroGNN 2021-08-16: 100%|██████████| 30/30 [02:05<00:00,  4.17s/it]
Оценка HeteroGNN 2021-08-16: 112it [00:05, 21.51it/s]


,week,model,val_groups,map10,ndcg10,best_epoch
0,2021-08-16,HeteroGNN,111240,0.078109,0.111970,30
1,2021-08-16,RRF,111240,0.047603,0.079496,30


In [24]:
baseline_dir = Path('results/lgbm_cv_weekly5000_full500_sliding6')
for filename in ('cv_results.csv', 'test_results.csv'):
    tables = []
    for name, folder in model_dirs.items():
        result = pd.read_csv(folder / filename)
        tables.append(result.query('model == @name').set_index('week')['map10'].rename(name))
    baseline_file = baseline_dir / filename
    if baseline_file.exists():
        baseline = pd.read_csv(baseline_file)
        for name in ('RRF', 'Basic', 'Manual'):
            tables.append(baseline.query('model == @name').set_index('week')['map10'].rename(name))
    comparison = pd.concat(tables, axis=1)
    print(filename)
    display(comparison.round(5))
    display(comparison.mean().round(5))

cv_results.csv


,LightGCN,HeteroGNN,RRF,Basic,Manual
week,,,,,
2021-06-21,0.12533,0.11801,0.06084,0.14912,0.16610
2021-06-28,0.18323,0.15747,0.06214,0.19703,0.25916
2021-07-05,0.15135,0.14891,0.06038,0.16865,0.13092
2021-07-12,0.11304,0.11129,0.05332,0.11767,0.14214
2021-07-19,0.09564,0.09494,0.05294,0.11112,0.12076
2021-07-26,0.09074,0.08961,0.05434,0.11225,0.11993
2021-08-02,0.08818,0.08766,0.05463,0.10950,0.11774


LightGCN     0.12107
HeteroGNN    0.11541
RRF          0.05694
Basic        0.13791
Manual       0.15096
dtype: float64

test_results.csv


,LightGCN,HeteroGNN,RRF,Basic,Manual
week,,,,,
2021-08-09,0.08118,0.08165,0.04999,0.09548,0.10669
2021-08-16,0.07761,0.07811,0.04760,0.09019,0.10426


LightGCN     0.07939
HeteroGNN    0.07988
RRF          0.04880
Basic        0.09284
Manual       0.10548
dtype: float64